In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_2016_2022_bn.csv")

In [3]:
df_de = df_de.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_de

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,DI,COC,CPI,ROL,VAA,EUPC
0,1.64,5.6,5.900000,1.31,236.000000,0.81,26.22,19.734865,19.519072,1.472293,...,73.6,0.43,-682.000000,3007.000000,8.63,1.78,81.0,1.58,1.36,2.350574e+06
1,2.89,5.5,5.800000,1.28,237.000000,0.37,26.12,20.513282,19.459550,2.332866,...,74.3,0.27,27047.000000,-17289.000000,8.61,1.80,81.0,1.57,1.43,1.710234e+07
2,4.00,5.5,5.600000,1.25,237.000000,0.30,25.94,21.470536,19.502234,0.813017,...,75.0,0.22,8209.000000,-28666.000000,8.68,1.90,80.0,1.59,1.44,2.260392e+07
3,1.91,5.3,5.500000,1.20,238.000000,0.23,25.63,21.300072,19.903024,0.760402,...,75.7,0.37,16219.010000,-50838.000000,8.68,1.87,80.0,1.58,1.35,2.762046e+07
4,4.49,5.5,6.166923,1.22,238.000000,0.08,25.38,21.666734,21.731321,-4.173453,...,74.6,-0.61,12158.990000,-58901.000000,8.67,1.83,80.0,1.52,1.37,2.586161e+07
5,2.93,5.5,6.166923,1.25,238.000000,0.04,25.31,22.461389,21.674219,3.626129,...,75.9,-0.09,-21616.800000,-17183.714286,8.67,1.78,80.0,1.57,1.42,3.239572e+07
6,2.09,5.5,6.166923,1.24,234.933333,0.72,25.69,23.021055,21.521327,0.641609,...,77.0,0.67,10577.546667,-17183.714286,8.80,1.82,79.0,1.53,1.41,3.509775e+07


In [5]:
discretised_de = pd.DataFrame(index=df_de.index)

for col in df_de.columns:
    no_unique = df_de[col].nunique()
    
    if no_unique <= 1:
        discretised_de[col] = 0
    else:
        try:
            discretised_de[col] = pd.qcut(
                df_de[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_de[col] = df_de[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_de)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    0    1    1     2     0    2     2     0     0      1  ...    0    2   
1    1    0    0     2     0    1     2     0     0      2  ...    0    1   
2    2    0    0     1     0    1     1     1     0      1  ...    1    0   
3    0    0    0     0     1    0     0     0     1      0  ...    1    1   
4    2    0    1     0     1    0     0     1     2      0  ...    0    0   
5    1    0    1     1     1    0     0     2     2      2  ...    2    0   
6    0    0    1     0     0    2     1     2     1      0  ...    2    2   

   EPFRG  GGAG  DI  COC  CPI  ROL  VAA  EUPC  
0      0     2   0    0    1    1    0     0  
1      2     1   0    0    1    0    2     0  
2      0     0   1    2    0    2    2     0  
3      2     0   1    2    0    1    0     1  
4      1     0   0    1    0    0    0     1  
5      0     1   0    0    0    0    1     2  
6      1     1   2    1    0    0   

In [6]:
sm = from_pandas(discretised_de)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_de.html")

Graphs/01_fully_connected_de.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_de.html")

Graphs/01_thresholded_de.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_de_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

    From     To    Weight
47   CPI  EPFRG  6.227985
41   CPI    EII  3.701716
14  PDPS    FDI  2.640454
46   CPI    LFG  2.503307
40   CPI   GGFC  2.175145
44   CPI    LTU  1.999966
17  VABI    PGP  1.988678
13  EIAP  GDPCG  1.922752
18  EBSG   LBGD  1.751709
36    DI    EMP  1.700086
12   QOA   GGFC  1.563268
21   LTU    FDI  1.506717
52   ROL    FDI  1.460254
48   CPI   GGAG  1.402028
1    QOR    QOA  1.263564
20  LBGD    EIS  1.194913
32   LTU    ROL  1.134748
22   LTU   EIAP  1.046476
33   LTU    VAA  1.006704
2    QOR    PGP  1.000366
9    QOR    CPI  0.999919
10   QOR    ROL  0.996250
31   LTU     DI  0.982126
7    QOR   GGAG  0.959028
24   LTU   VABI  0.952279
15   PGP    EIS  0.919661
35  GGAG   EUPC  0.893615
27   LTU   EBSG  0.842175
34  GGAG    EMP  0.812413
55  EUPC   GDPC  0.811816
42   CPI    EIS -0.811319
54   ROL    EIS -0.874755
26   LTU  GDPCG -0.894634
25   LTU   GGFC -0.955040
8    QOR     DI -0.964785
23   LTU    PGP -0.984905
4    QOR  GDPCG -0.988400
38   CPI    